<a href="https://colab.research.google.com/github/iav2002/AppliedDeepLearning/blob/main/Part2_8_OwnVariation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2, Notebook 8, Own Variation

Best model per task by stacking what worked across the assignment. Skip connections from notebook 5, augmentation from notebook 7, plus three techniques not tested yet, batch normalisation, light dropout in the head (AlexNet reference), and AdamW with cosine annealed learning rate. Same template for regression and classification, parameterised by kernel size and output head.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Colab Notebooks/AppliedDL/face_age.zip" /content/
!cp -r "/content/drive/MyDrive/Colab Notebooks/AppliedDL/data_splits" /content/
!unzip -q /content/face_age.zip -d /content/

## 2. Imports

In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

cuda
gpu: NVIDIA A100-SXM4-40GB


## 3. Dataset class

In [4]:
CATEGORIES = ["infant", "child", "teen", "youth", "mid", "mature", "senior"]
CAT_TO_IDX = {c: i for i, c in enumerate(CATEGORIES)}


class FaceAgeDataset(Dataset):
    def __init__(self, csv_path, task, transform=None):
        self.df = pd.read_csv(csv_path)
        self.task = task
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"/content/{row['path']}").convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.task == "reg":
            lbl = torch.tensor(row["age"], dtype=torch.float32)
        else:
            lbl = torch.tensor(CAT_TO_IDX[row["age_category"]], dtype=torch.long)
        return img, lbl

## 4. Train and eval functions, supports both tasks

In [8]:
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    model.train()
    total_loss = 0
    n_samples = 0

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        preds = model(imgs).squeeze()
        loss = loss_fn(preds, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        n_samples += imgs.size(0)
    if scheduler is not None:
        scheduler.step()
    return total_loss / n_samples


def evaluate(model, loader, loss_fn, task):
    model.eval()
    total_loss = 0
    n_samples = 0
    correct = 0
    abs_err = 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            preds = model(imgs).squeeze()
            loss = loss_fn(preds, lbls)
            total_loss += loss.item() * imgs.size(0)
            n_samples += imgs.size(0)
            if task == "reg":
                abs_err += (preds - lbls).abs().sum().item()
            else:
                correct += (preds.argmax(1) == lbls).sum().item()
    avg_loss = total_loss / n_samples
    metric = abs_err / n_samples if task == "reg" else correct / n_samples
    return avg_loss, metric


def run_variant(model, train_loader, val_loader, loss_fn, task,
                max_epochs=20, patience=5, lr=1e-3, weight_decay=1e-4, verbose=True):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-5)

    if task == "reg":
        best_metric = float("inf")
        better = lambda new, best: new < best
    else:
        best_metric = -float("inf")
        better = lambda new, best: new > best

    best_state = None
    history = {"train_loss": [], "val_loss": [], "val_metric": [], "lr": []}
    no_improve = 0

    for epoch in range(max_epochs):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, scheduler)
        val_loss, val_metric = evaluate(model, val_loader, loss_fn, task)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_metric"].append(val_metric)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        if verbose:
            tag = "MAE" if task == "reg" else "acc"
            print(f"epoch {epoch+1:2d}  train {train_loss:.4f}  val {val_loss:.4f}  {tag} {val_metric:.4f}  lr {optimizer.param_groups[0]['lr']:.2e}")

        if better(val_metric, best_metric):
            best_metric = val_metric
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                if verbose:
                    print(f"early stop at epoch {epoch+1}")
                break

    return {"best_metric": best_metric, "best_state": best_state, "history": history}

## 5. Model

Same architecture template for both tasks, parameterised by kernel size and output head. Three conv blocks with skip connections (b3 style from notebook 5, two convs per block plus an optional 1x1 projection residual). BatchNorm after every conv. Dropout 0.2 before the final linear in the head, AlexNet style regularisation pairing well with augmentation. ReLU as established in notebook 4.

In [9]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size, padding=pad)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size, padding=pad)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.proj = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        identity = self.proj(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        out = self.pool(out)
        return out


class BestModel(nn.Module):
    def __init__(self, task, kernel_size=3):
        super().__init__()
        self.task = task
        self.features = nn.Sequential(
            ResBlock(3, 32, kernel_size),
            ResBlock(32, 64, kernel_size),
            ResBlock(64, 128, kernel_size),
        )
        out_dim = 1 if task == "reg" else 7
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 25 * 25, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, out_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x

## 6. Transforms

Same moderate augmentation preset from notebook 7 on train, clean transform on validation. ImageNet normalisation.

In [10]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

tf_train = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

tf_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

## 7. Run, regression

In [11]:
train_ds = FaceAgeDataset("/content/data_splits/train.csv", task="reg", transform=tf_train)
val_ds = FaceAgeDataset("/content/data_splits/val.csv", task="reg", transform=tf_val)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

print("=== regression, best model ===")
t0 = time.time()

model = BestModel(task="reg", kernel_size=5).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"params {n_params:,}")

loss_fn = nn.MSELoss()
out_reg = run_variant(model, train_loader, val_loader, loss_fn, task="reg")
out_reg["time"] = time.time() - t0
print(f"best MAE {out_reg['best_metric']:.4f}  time {out_reg['time']:.1f}s")

=== regression, best model ===
params 11,048,161
epoch  1  train 935.1291  val 487.6711  MAE 17.3962  lr 9.94e-04
epoch  2  train 564.6509  val 358.5895  MAE 13.3835  lr 9.76e-04
epoch  3  train 490.4257  val 215.6388  MAE 11.0402  lr 9.46e-04
epoch  4  train 450.5159  val 174.5778  MAE 9.6703  lr 9.05e-04
epoch  5  train 421.2016  val 295.2277  MAE 12.5306  lr 8.55e-04
epoch  6  train 386.0971  val 202.2672  MAE 10.1748  lr 7.96e-04
epoch  7  train 387.7267  val 178.7097  MAE 9.4556  lr 7.30e-04
epoch  8  train 402.3193  val 192.8745  MAE 9.9502  lr 6.58e-04
epoch  9  train 380.1247  val 271.8418  MAE 12.1160  lr 5.82e-04
epoch 10  train 367.8814  val 204.0918  MAE 10.2997  lr 5.05e-04
epoch 11  train 368.1357  val 176.4278  MAE 9.5356  lr 4.28e-04
epoch 12  train 355.2118  val 161.1845  MAE 8.9704  lr 3.52e-04
epoch 13  train 354.9302  val 122.7994  MAE 7.7786  lr 2.80e-04
epoch 14  train 351.2692  val 112.3014  MAE 7.3638  lr 2.14e-04
epoch 15  train 364.4690  val 133.6898  MAE 8.07

## 8. Run, classification

In [12]:
train_ds = FaceAgeDataset("/content/data_splits/train.csv", task="cls", transform=tf_train)
val_ds = FaceAgeDataset("/content/data_splits/val.csv", task="cls", transform=tf_val)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

print("=== classification, best model ===")
t0 = time.time()

model = BestModel(task="cls", kernel_size=3).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"params {n_params:,}")

loss_fn = nn.CrossEntropyLoss()
out_cls = run_variant(model, train_loader, val_loader, loss_fn, task="cls")
out_cls["time"] = time.time() - t0
print(f"best acc {out_cls['best_metric']:.4f}  time {out_cls['time']:.1f}s")

=== classification, best model ===
params 10,539,495
epoch  1  train 3.7759  val 1.6152  acc 0.3607  lr 9.94e-04
epoch  2  train 1.5548  val 1.3975  acc 0.4242  lr 9.76e-04
epoch  3  train 1.3989  val 1.1950  acc 0.5034  lr 9.46e-04
epoch  4  train 1.2786  val 1.1429  acc 0.5444  lr 9.05e-04
epoch  5  train 1.1979  val 1.1973  acc 0.5061  lr 8.55e-04
epoch  6  train 1.1667  val 1.0323  acc 0.5813  lr 7.96e-04
epoch  7  train 1.1072  val 1.0407  acc 0.5813  lr 7.30e-04
epoch  8  train 1.0968  val 0.9432  acc 0.6086  lr 6.58e-04
epoch  9  train 1.0347  val 0.9157  acc 0.6277  lr 5.82e-04
epoch 10  train 1.0050  val 0.9455  acc 0.6066  lr 5.05e-04
epoch 11  train 0.9678  val 0.8982  acc 0.6264  lr 4.28e-04
epoch 12  train 0.9459  val 0.8654  acc 0.6475  lr 3.52e-04
epoch 13  train 0.9240  val 0.8373  acc 0.6516  lr 2.80e-04
epoch 14  train 0.8895  val 0.8211  acc 0.6462  lr 2.14e-04
epoch 15  train 0.8739  val 0.8128  acc 0.6571  lr 1.55e-04
epoch 16  train 0.8470  val 0.7982  acc 0.6598 